In [23]:
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

Opdracht 5

Opdracht 5.a 

Reset Database

In [24]:
aenc_sdm_connection = sqlite3.connect('sdm.db')

alle_tabellen = [
    'Accessoire_inkoop'
    'Accessoire_inkoop_Accessoire'
    'Accessoire_inkoop_Leverancier'
    'Accessoireverkoop'
    'Accessoireverkoop_Accessoire'
    'Accessoireverkoop_Leverancier'
    'Accessoireverkoop_Monteur'
    'Accessoireverkoop_filiaal'
    'Accessoireverkoop_klant'
    'Fiets_Inkoop'
    'Fiets_Inkoop_Fabrikant'
    'Fiets_Inkoop_Fiets'
    'Fietsverkoop'
    'Fietsverkoop_Fabrikant'
    'Fietsverkoop_Fiets'
    'Fietsverkoop_Monteur'
    'Fietsverkoop_filiaal'
    'Fietsverkoop_klant'
    'Onderhoud'
    'Onderhoud_Fabrikant'
    'Onderhoud_Fiets'
    'Onderhoud_filiaal'
    'Onderhoud_monteur'
]

def reset_sdm():
    aenc_sdm_cursor = aenc_sdm_connection.cursor()
    for tabel in alle_tabellen:
        delete_statement = f"DELETE FROM {tabel};"

        try:
            aenc_sdm_cursor.execute(delete_statement)
        except pyodbc.Error:
            print(f"FAILED: {delete_statement}")
            continue
    aenc_sdm_connection.commit()

In [25]:
sdm_connection = sqlite3.connect('sdm.db')
cursor = sdm_connection.cursor()

def reset_sdm ():
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';", sdm_connection)

    for table in tables['name']:
        cursor.execute(f"DROP TABLE IF EXISTS {table}")

    sdm_connection.commit()
    print("Database geleegt completed.")

def load_sdm():
    df = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", sdm_connection)

    df.to_sql('sdm_tables', sdm_connection, if_exists='replace', index=False)

    print("Data geladen in SDM.")

def full_refresh_sdm():
    reset_sdm()
    load_sdm()
    print("Full refresh van SDM completed.")

Opdracht 5.b



In [26]:
full_refresh_sdm()



Database geleegt completed.
Data geladen in SDM.
Full refresh van SDM completed.


In [27]:
import os

dbs = [
    'BikeToDrive_1_Accessoireverkoop.db',
    'BikeToDrive_2_Fietsverkoop.db',
    'BikeToDrive_3_Onderhoud.db',
    'BikeToDrive_4_Accessoire_Inkoop.db',
    'BikeToDrive_5_Fiets_Inkoop.db'
]

for db in dbs:
    if os.path.exists(db):
        conn = sqlite3.connect(db)
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", conn)
        print(f'{db}: {list(tables["name"])}')
        conn.close()
    else:
        print(f'{db} not found')

BikeToDrive_1_Accessoireverkoop.db: ['Accessoire_Verkoop', 'Monteur', 'Leverancier', 'Accessoire', 'Filiaal', 'Klant']
BikeToDrive_2_Fietsverkoop.db: ['Fiets_Verkoop', 'Fiets', 'Monteur', 'Fabrikant', 'Filiaal', 'Klant']
BikeToDrive_3_Onderhoud.db: ['Onderhoud', 'Fiets', 'Monteur', 'Fabrikant', 'Filiaal']
BikeToDrive_4_Accessoire_Inkoop.db: ['Accessoire_Inkoop', 'Accessoire', 'Leverancier']
BikeToDrive_5_Fiets_Inkoop.db: ['Fiets_Inkoop', 'Fiets', 'Fabrikant']


In [28]:
def load_data_to_sdm():
    sdm_conn = sqlite3.connect('sdm.db')

    dbs = [
        'BikeToDrive_1_Accessoireverkoop.db',
        'BikeToDrive_2_Fietsverkoop.db',
        'BikeToDrive_3_Onderhoud.db',
        'BikeToDrive_4_Accessoire_Inkoop.db',
        'BikeToDrive_5_Fiets_Inkoop.db'
    ]
    table_mapping = {
        'Accessoire_Verkoop': 'Accessoireverkoop',
        'Fiets_Verkoop': 'Fietsverkoop',
        'Accessoire_Inkoop': 'Accessoire_inkoop',
        'Fiets_Inkoop': 'Fiets_Inkoop',
    }

    for db in dbs:
        if not os.path.exists(db):
            print(f'{db} not found')
            continue
        conn = sqlite3.connect(db)
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'", conn)
        for table in tables['name']:
            df = pd.read_sql(f"SELECT * FROM {table}", conn)
            sdm_table = table_mapping.get(table, table)
            df.to_sql(sdm_table, sdm_conn, if_exists='append', index=False)
            print(f'Loaded {len(df)} rows from {table} in {db} to {sdm_table}')
        conn.close()

    sdm_conn.close()
    print("Data loading completed.")


In [29]:
shared_tables = ['Klant', 'Monteur', 'Filiaal', 'Leverancier', 'Accessoire', 'Fiets', 'Fabrikant']

for table in shared_tables:
    ids = set()
    for db in dbs:
        if not os.path.exists(db):
            continue
        conn = sqlite3.connect(db)
        try:
            df = pd.read_sql(f"SELECT id FROM {table}", conn)
            ids.update(df['id'].tolist())
        except:
            pass
        conn.close()
    print(f'{table}: {len(ids)} unique IDs')

Klant: 0 unique IDs
Monteur: 0 unique IDs
Filiaal: 0 unique IDs
Leverancier: 0 unique IDs
Accessoire: 0 unique IDs
Fiets: 0 unique IDs
Fabrikant: 0 unique IDs


In [30]:
conn = sqlite3.connect('BikeToDrive_1_Accessoireverkoop.db')
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(Klant)")
columns = cursor.fetchall()
print("Klant columns:", [col[1] for col in columns])
df = pd.read_sql("SELECT * FROM Klant LIMIT 5", conn)
print(df.head())
conn.close()

Klant columns: ['klantnr', 'naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum']
   klantnr            naam           adres woonplaats geslacht geboortedatum
0        1      Jan Jansen   Kerkstraat 12  Amsterdam        M    1985-03-22
1        2  Sophie de Boer     Lindelaan 8  Rotterdam        V    1990-07-11
2        3   Pieter Visser   Havenstraat 3   Den Haag        M    1978-11-05
3        4       Emma Smit    Boomgaard 22    Haarlem        V    1995-02-18
4        5      Tom Bakker  Stationsweg 44     Leiden        M    1982-09-09


In [31]:
conn = sqlite3.connect('BikeToDrive_2_Fietsverkoop.db')
df = pd.read_sql("SELECT * FROM Klant LIMIT 5", conn)
print("Fietsverkoop Klant:")
print(df.head())
conn.close()

Fietsverkoop Klant:
   klantnr            naam           adres woonplaats geslacht geboortedatum
0        1      Jan Jansen   Kerkstraat 12  Amsterdam        M    1985-03-22
1        2  Sophie de Boer     Lindelaan 8  Rotterdam        V    1990-07-11
2        3   Pieter Visser   Havenstraat 3   Den Haag        M    1978-11-05
3        4       Emma Smit    Boomgaard 22    Haarlem        V    1995-02-18
4        5      Tom Bakker  Stationsweg 44     Leiden        M    1982-09-09


In [32]:
with open('BikeToDrive_RIM - alle bestanden.txt', 'r', encoding='utf-8') as f:
    sql = f.read()

sdm_conn = sqlite3.connect('sdm.db')
cursor = sdm_conn.cursor()

statements = [stmt.strip() for stmt in sql.split(';') if stmt.strip() and not stmt.strip().startswith('--')]

for stmt in statements:
    try:
        cursor.execute(stmt)
        print(f"Executed: {stmt[:50]}...")
    except Exception as e:
        print(f"Error: {e} for {stmt[:50]}")

sdm_conn.commit()
sdm_conn.close()
print("Tables created in SDM.")

Executed: CREATE TABLE Fietsverkoop_Fabrikant (
	fabrikantnr...
Executed: CREATE TABLE Fietsverkoop_filiaal (
	filiaalnr INT...
Executed: CREATE TABLE Fietsverkoop_klant (
	klantnr INT,
	n...
Executed: CREATE TABLE Fietsverkoop_Monteur (
	monteurnr INT...
Executed: CREATE TABLE Fietsverkoop_Fiets (
	fietsnr INT,
	s...
Executed: CREATE TABLE Fietsverkoop (
	fiets_verkoopnr INT,
...
Executed: CREATE TABLE Accessoireverkoop_Leverancier (
	leve...
Executed: CREATE TABLE Accessoireverkoop_filiaal (
	filiaaln...
Executed: CREATE TABLE Accessoireverkoop_klant (
	klantnr IN...
Executed: CREATE TABLE Accessoireverkoop_Monteur (
	monteurn...
Executed: CREATE TABLE Accessoireverkoop_Accessoire (
	acces...
Executed: CREATE TABLE Accessoireverkoop (
	accessoire_verko...
Executed: CREATE TABLE Accessoire_Inkoop_Leverancier (
	leve...
Executed: CREATE TABLE Accessoire_Inkoop_Accessoire (
	acces...
Executed: CREATE TABLE Accessoire_Inkoop (
	inkoopnr INT,
	i...
Executed: CREATE TABLE Fiets_Inkoop_Fabr

In [33]:
mappings = {
    'BikeToDrive_1_Accessoireverkoop.db': {
        'Accessoire_Verkoop': 'Accessoireverkoop',
        'Monteur': 'Accessoireverkoop_Monteur',
        'Leverancier': 'Accessoireverkoop_Leverancier',
        'Accessoire': 'Accessoireverkoop_Accessoire',
        'Filiaal': 'Accessoireverkoop_filiaal',
        'Klant': 'Accessoireverkoop_klant'
    },
    'BikeToDrive_2_Fietsverkoop.db': {
        'Fiets_Verkoop': 'Fietsverkoop',
        'Fiets': 'Fietsverkoop_Fiets',
        'Monteur': 'Fietsverkoop_Monteur',
        'Fabrikant': 'Fietsverkoop_Fabrikant',
        'Filiaal': 'Fietsverkoop_filiaal',
        'Klant': 'Fietsverkoop_klant'
    },
    'BikeToDrive_3_Onderhoud.db': {
        'Onderhoud': 'Onderhoud',
        'Fiets': 'Onderhoud_Fiets',
        'Monteur': 'Onderhoud_Monteur',
        'Fabrikant': 'Onderhoud_Fabrikant',
        'Filiaal': 'Onderhoud_filiaal'
    },
    'BikeToDrive_4_Accessoire_Inkoop.db': {
        'Accessoire_Inkoop': 'Accessoire_Inkoop',
        'Accessoire': 'Accessoire_Inkoop_Accessoire',
        'Leverancier': 'Accessoire_Inkoop_Leverancier'
    },
    'BikeToDrive_5_Fiets_Inkoop.db': {
        'Fiets_Inkoop': 'Fiets_Inkoop',
        'Fiets': 'Fiets_Inkoop_Fiets',
        'Fabrikant': 'Fiets_Inkoop_Fabrikant'
    }
}

sdm_conn = sqlite3.connect('sdm.db')

for db, table_map in mappings.items():
    if not os.path.exists(db):
        print(f'{db} not found')
        continue
    conn = sqlite3.connect(db)
    for src_table, dst_table in table_map.items():
        try:
            df = pd.read_sql(f"SELECT * FROM {src_table}", conn)
            df.to_sql(dst_table, sdm_conn, if_exists='append', index=False)
            print(f'Loaded {len(df)} rows from {src_table} to {dst_table}')
        except Exception as e:
            print(f'Error loading {src_table}: {e}')
    conn.close()

sdm_conn.close()
print("Data loading completed.")

Loaded 100 rows from Accessoire_Verkoop to Accessoireverkoop
Loaded 10 rows from Monteur to Accessoireverkoop_Monteur
Loaded 5 rows from Leverancier to Accessoireverkoop_Leverancier
Loaded 10 rows from Accessoire to Accessoireverkoop_Accessoire
Loaded 4 rows from Filiaal to Accessoireverkoop_filiaal
Loaded 20 rows from Klant to Accessoireverkoop_klant
Loaded 150 rows from Fiets_Verkoop to Fietsverkoop
Loaded 75 rows from Fiets to Fietsverkoop_Fiets
Loaded 10 rows from Monteur to Fietsverkoop_Monteur
Loaded 10 rows from Fabrikant to Fietsverkoop_Fabrikant
Loaded 4 rows from Filiaal to Fietsverkoop_filiaal
Loaded 25 rows from Klant to Fietsverkoop_klant
Loaded 50 rows from Onderhoud to Onderhoud
Loaded 30 rows from Fiets to Onderhoud_Fiets
Loaded 15 rows from Monteur to Onderhoud_Monteur
Loaded 11 rows from Fabrikant to Onderhoud_Fabrikant
Error loading Filiaal: table "Onderhoud_filiaal" already exists
Loaded 50 rows from Accessoire_Inkoop to Accessoire_Inkoop
Loaded 13 rows from Accesso